In [11]:
from datasets import load_dataset

# dataset = load_dataset(
#     "HuggingFaceH4/ultrachat_200k",
#     cache_dir="/code/ultrachat_200k",
# )
# dataset = dataset["train_sft"]
# dataset = load_dataset(
#     "allenai/tulu-3-sft-mixture",
#     cache_dir="/code/datasets/tulu-3-sft-mixture",
# )
# dataset = dataset["train"]
dataset = load_dataset(
    "zai-org/LongAlign-10k",
    cache_dir="/code/datasets/LongAlign-10k",
    # num_proc=5,
)
dataset = dataset["train"]

README.md: 0.00B [00:00, ?B/s]

long.jsonl:   0%|          | 0.00/655M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9888 [00:00<?, ? examples/s]

In [4]:
dataset = dataset.filter(
    lambda example: not "gpt-3.5" in example["model"],
    num_proc=22,
)

Filter (num_proc=22):   0%|          | 0/3199860 [00:00<?, ? examples/s]

In [12]:
dataset[0]["conversation"]

KeyError: 'conversation'

In [ ]:
dataset[0]["conversations"]

In [ ]:
dataset[0]["messages"]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name)

text = tokenizer.apply_chat_template(
    [
        dataset[0]["messages"],
        dataset[1]["messages"],
    ],
    tokenize=False,
    add_generation_prompt=False,
)
print(text)
# len(
#     tokenizer.encode(
#         text,
#         add_special_tokens=False,
#     )
# )

In [ ]:
tokenizer(
    text,
    padding=True,
    return_tensors="pt",
)["input_ids"].shape

torch.Size([2, 1479])

In [5]:
len(text)

2

In [15]:
dataset[0]

{'conversations': [{'from': 'system',
   'value': 'You are a helpful assistant, who always provide explanation. Think like you are answering to a five year old.'},
  {'from': 'user',
   'value': 'Arabian jääkiekkoliitto\n\nTranslate this to English?'},
  {'from': 'gpt',
   'value': 'OK, let\'s translate that for you, little one!\n\n"Arabian jääkiekkoliitto" is not in English. It\'s in Finnish, a language spoken in a country called Finland.\n\nIn English, it means "Arab Ice Hockey Federation."\n\nNow, let me explain what that means:\n\n1. "Arab" refers to people or things from countries in the Middle East and North Africa.\n2. "Ice Hockey" is a fun sport played on ice where players use sticks to hit a small rubber puck into a goal.\n3. "Federation" is like a big group of people who work together for the same purpose.\n\nSo, the Arab Ice Hockey Federation is a group that helps organize and promote ice hockey in Arab countries. It\'s like a team of grown-ups who make sure kids and adults 

In [15]:
def get_len(item):
    if "messages" in item:
        messages = item["messages"]
    elif "conversations" in item:
        messages = []
        for mess in item["conversations"]:
            value = mess["value"]
            from_ = mess["from"]
            if from_ == "gpt":
                from_ = "assistant"
            messages.append(
                {
                    "role": from_,
                    "content": value,
                }
            )
    else:
        messages = []
        for mess in item["conversation"]:
            value = mess["content"]
            from_ = mess["role"]

            messages.append(
                {
                    "role": from_,
                    "content": value,
                }
            )

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    length = len(
        tokenizer.encode(
            text,
            add_special_tokens=False,
        )
    )
    return {"length": length}


dataset = dataset.map(get_len, num_proc=20)
pass

Map (num_proc=20):   0%|          | 0/9888 [00:00<?, ? examples/s]

In [16]:
dataset[0]["length"]

53315

In [19]:
import numpy as np

lens = dataset["length"]
mean_time = np.mean(lens)

# Calculate key percentiles
p50 = np.percentile(lens, 50)  # This is the median
p90 = np.percentile(lens, 90)
p95 = np.percentile(lens, 95)
p99 = np.percentile(lens, 99)

print(f"50th percentile (median): {p50:.2f} tokens")
print(f"90th percentile: {p90:.2f} tokens")
print(f"95th percentile: {p95:.2f} tokens")
print(f"99th percentile: {p99:.2f} tokens")

50th percentile (median): 12099.50 tokens
90th percentile: 33515.10 tokens
95th percentile: 43141.00 tokens
99th percentile: 55624.71 tokens


In [ ]:
# ultrachat - 2229
# tulu-3-sft-mixture - 2019.00
# Sonnet3.5-SlimOrcaDedupCleaned - 1093
# WildChat-4.8M - 19662.00
# zai-org/LongAlign-10k - 43141.00

In [20]:
len(dataset) * 0.95

9393.6

In [ ]:
2229.00 // 64

34.0

In [ ]:
35.0 * 64

2240.0

In [26]:
dataset = dataset.filter(lambda example: example["length"] < 2240)

Filter:   0%|          | 0/207865 [00:00<?, ? examples/s]

197748

### Load model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from train_gym.rmt.rmt_wrappers import (
    MemoryCell,
    RecurrentWrapper,
    MemoryCellTrain,
    RecurrentWrapperTrain,
    MemoryCellTrainLiger,
    lce_forward,
)

# model_name = "Qwen/Qwen3-4B-Instruct-2507"
model_name = "Qwen/Qwen3-1.7B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)


peft_config = get_peft_config(model_args)
model = get_peft_model(model, peft_config)

print("apply_rmt")
# посегментное вычисление лосса
cell = MemoryCellTrain(
    model,
    num_mem_tokens=rmt_args.memory_size,
)
model = RecurrentWrapperTrain(
    cell,
    segment_size=rmt_args.segment_size,
    max_n_segments=rmt_args.max_n_segments,
    vary_n_segments=rmt_args.vary_n_segments,
    k2=rmt_args.k2,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

loading file vocab.json from cache at /home/user-name-goes-here/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/vocab.json
loading file merges.txt from cache at /home/user-name-goes-here/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/merges.txt
loading file tokenizer.json from cache at /home/user-name-goes-here/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /home/user-name-goes-here/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained

...

model-00002-of-00002.safetensors:   0%|          | 0.00/622M [00:00<?, ?B/s]

Will use dtype=torch.bfloat16 as defined in model's config object
Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645
}



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loading configuration file generation_config.json from cache at /home/user-name-goes-here/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "temperature": 0.6,
  "top_k": 20,
  "top_p": 0.95
}

Could not locate the custom_generate/generate.py inside Qwen/Qwen3-1.7B.


In [ ]:
# model.save_pretrained('model_checkpoints/Qwen3-4B-Instruct-2507')
# tokenizer.save_pretrained('model_checkpoints/Qwen3-4B-Instruct-2507')
model.save_pretrained("model_checkpoints/Qwen3-1.7B")
tokenizer.save_pretrained("model_checkpoints/Qwen3-1.7B")

Configuration saved in model_checkpoints/Qwen3-1.7B/config.json
Configuration saved in model_checkpoints/Qwen3-1.7B/generation_config.json
Model weights saved in model_checkpoints/Qwen3-1.7B/model.safetensors
chat template saved in model_checkpoints/Qwen3-1.7B/chat_template.jinja
tokenizer config file saved in model_checkpoints/Qwen3-1.7B/tokenizer_config.json
Special tokens file saved in model_checkpoints/Qwen3-1.7B/special_tokens_map.json


('model_checkpoints/Qwen3-1.7B/tokenizer_config.json',
 'model_checkpoints/Qwen3-1.7B/special_tokens_map.json',
 'model_checkpoints/Qwen3-1.7B/chat_template.jinja',
 'model_checkpoints/Qwen3-1.7B/vocab.json',
 'model_checkpoints/Qwen3-1.7B/merges.txt',
 'model_checkpoints/Qwen3-1.7B/added_tokens.json',
 'model_checkpoints/Qwen3-1.7B/tokenizer.json')

In [ ]:
from transformers.models.qwen3.modeling_qwen3 import Qwen3DecoderLayer